# 03 -- Collections: 50 Songs About a Keyword

Returns **50 globally most-played songs** matching a given keyword (love, war, happiness, loneliness, money),
using lyrics-based content filtering.

### Algorithm

Three approaches are compared:

1. **Baseline** -- exact keyword match: count occurrences of the keyword in each
   track's lyrics, apply a threshold, then sort by total play count.
2. **Word2Vec** -- expand the keyword with semantically similar tokens via a
   pre-trained word2vec model, then combine their lyric counts.
3. **Classification** -- label tracks as "about X" vs "not about X" using
   keyword presence, train a classifier on the labelled set, then predict
   scores for the remaining tracks.

### Output columns
| Column | Description |
|--------|-------------|
| `rank` | 1-50 index, by total play count (descending) |
| `artist` | Artist name |
| `title` | Track title |
| `play_count` | Total plays across all users |

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.data import MySpotifyRecommender
from src.models.collections import (
    collection_baseline,
    collection_classification_compare,
    collection_word2vec,
)

In [2]:
rs = MySpotifyRecommender.from_files(Path.cwd().parent / "data", download=True)

  tracks      (1000000, 4)
  genres      (280831, 3)
  triplets    (48373586, 3)
  lyrics_long (16845943, 3)


---
## Research

In [3]:
KEYWORDS = ["love", "war", "happiness", "loneliness", "money"]

### Approach 1 -- Baseline (exact keyword match)

In [4]:
baseline_results = collection_baseline(rs, KEYWORDS, n=1, top_n=50)
for i in range(len(KEYWORDS)):
    keyword = KEYWORDS[i]
    result = baseline_results.get(keyword, "No results found")
    print(f"Keyword: {keyword}")
    display(result)
    print("\n")

Keyword: love


index,artist,title,play_count
u32,str,str,i64
0,"""Dwight Yoakam""","""You're The One""",726885
1,"""Tub Ring""","""Invalid""",268353
2,"""Sam Cooke""","""Ain't Misbehavin""",244730
3,"""Train""","""Hey_ Soul Sister""",209212
4,"""Train""","""Marry Me""",174080
…,…,…,…
45,"""Chiddy Bang""","""Opposite Of Adults""",34504
46,"""Nick Cave & The Bad Seeds""","""Spell""",33624
47,"""The Black Keys""","""All Hands Against His Own""",33178




Keyword: war


index,artist,title,play_count
u32,str,str,i64
0,"""Sheena Easton""","""Strut (1993 Digital Remaster)""",115134
1,"""Eminem / Nate Dogg""","""'Till I Collapse""",44305
2,"""Creedence Clearwater Revival""","""Fortunate Son""",34506
3,"""O.G.C.""","""Gunn Clapp""",33897
4,"""John Mayer""","""Waiting On The World To Change""",24455
…,…,…,…
45,"""Rilo Kiley""","""Love And War (11/11/46) (Album…",6012
46,"""Rise Against""","""Life Less Frightening""",5944
47,"""Portugal. The Man""","""People Say""",5911




Keyword: happiness


index,artist,title,play_count
u32,str,str,i64
0,"""Sam Cooke""","""Ain't Misbehavin""",244730
1,"""Train""","""Marry Me""",174080
2,"""Radiohead""","""Creep (Explicit)""",98854
3,"""Counting Crows""","""Mr. Jones""",51287
4,"""Florence + The Machine""","""I'm Not Calling You A Liar""",40082
…,…,…,…
45,"""Boyz II Men""","""End Of The Road""",7760
46,"""The Temper Trap""","""Fools""",7749
47,"""Britt Nicole""","""You""",7734




Keyword: loneliness


index,artist,title,play_count
u32,str,str,i64
0,"""Radney Foster""","""The Kindness Of Strangers""",28115
1,"""Leona Lewis""","""Bleeding Love""",22195
2,"""The Black Keys""","""Too Afraid To Love""",12342
3,"""Despised Icon""","""In The Arms Of Perdition""",10624
4,"""Amos Lee""","""Black River""",9158
…,…,…,…
45,"""Cocorosie""","""Smokey Taboo""",1675
46,"""Charles Wright & The Watts 103…","""Love Land""",1614
47,"""Kings Of Convenience""","""My Ship Isn't Pretty""",1571




Keyword: money


index,artist,title,play_count
u32,str,str,i64
0,"""Beastie Boys""","""Unite (2009 Digital Remaster)""",99137
1,"""The Verve""","""Bitter Sweet Symphony""",76893
2,"""Eminem""","""Mockingbird""",74103
3,"""M.I.A.""","""Paper Planes""",50763
4,"""Eminem / Nate Dogg""","""'Till I Collapse""",44305
…,…,…,…
45,"""P. Diddy""","""I Need A Girl (Part One) (Feat…",8824
46,"""Jackson Browne""","""The Pretender""",8601
47,"""The Isley Brothers""","""Contagious""",8575


### Approach 2 -- Word2Vec (expanded keywords)

In [5]:
w2v_results = collection_word2vec(rs, KEYWORDS, n=50, top_n=50)
for kw, df in w2v_results.items():
    print(f"\n{'='*50}")
    print(f"  Collection: {kw.upper()}")
    print(f"{'='*50}")
    display(df)


  Collection: LOVE


index,artist,title,play_count
u32,str,str,i64
0,"""N.E.R.D.""","""Rock Star""",52834
1,"""Eminem / Dina Rae""","""Superman""",45328
2,"""Eminem / Dina Rae""","""Superman""",45328
3,"""Eminem / Nate Dogg""","""'Till I Collapse""",44305
4,"""Guns N' Roses""","""Don't Cry (Original)""",40480
…,…,…,…
45,"""Scouting for Girls""","""She's So Lovely""",8077
46,"""T-Pain featuring Teddy Verseti""","""Church""",8034
47,"""The Maine""","""I Wanna Love You (Akon Cover) …",7977



  Collection: WAR


index,artist,title,play_count
u32,str,str,i64
0,"""Frankie Goes To Hollywood""","""War""",213
1,"""Tiziano Ferro""","""Soul-dier""",115
2,"""Culture Club""","""The War Song (2003 Digital Rem…",0
3,"""Hot Boys""","""Introduction (Hot Boyz/Let Em …",0
4,"""Cheryl Cole""","""Fight For This Love""",0



  Collection: HAPPINESS


index,artist,title,play_count
u32,str,str,i64
0,"""Wilson Pickett""","""I'm In Love (Single/LP Version…",9713
1,"""Enrique Iglesias / Ciara""","""Takin' Back My Love""",3155
2,"""Jill Scott""","""It's Love""",2648
3,"""Zapp & Roger""","""Computer Love""",1896
4,"""Luther Vandross""","""Wait For Love""",1330
…,…,…,…
45,"""R. Kelly""","""You Made Me Love You""",4
46,"""Java""","""Don't Phunk With My Heart""",0
47,"""Karyn White""","""I'm Your Woman (Album Version)""",0



  Collection: LONELINESS


index,artist,title,play_count
u32,str,str,i64
0,"""Guy Forsyth""","""Brownsville""",5
1,"""Skip James""","""How Long 'Buck'""",0
2,"""David Bowie""","""Cat People (Putting Out Fire) …",0



  Collection: MONEY


index,artist,title,play_count
u32,str,str,i64
0,"""Black Eyed Peas""","""Let's Get It Started""",20511
1,"""Ok Go""","""Get Over It""",7639
2,"""50 Cent""","""I Get Money""",6140
3,"""50 Cent""","""I Get Money""",6140
4,"""En Vogue""","""My Lovin' (You're Never Gonna …",3093
…,…,…,…
45,"""Mr. Magic""","""Gangsta""",0
46,"""Jem""","""Keep On Walking (Album Version…",0
47,"""J2K""","""Hands Up""",0


### Approach 3 -- Classification (MultinomialNB / Logistic / SGD / RandomForest on lyrics)

In [6]:
classifiers = ["nb", "logistic", "sgd", "forest"]
clf_results = collection_classification_compare(rs, KEYWORDS, n=10, neg_ratio=1, classifiers=classifiers)
for name in classifiers:
    print(f"\n{'='*60}\nClassifier: {name.upper()}\n{'='*60}")
    for kw, df in clf_results[name].items():
        print(f"\n  Collection: {kw.upper()}")
        display(df)


Classifier: NB

  Collection: LOVE


index,artist,title,play_count
u32,str,str,i64
0,"""OneRepublic""","""Secrets""",292642
1,"""Tub Ring""","""Invalid""",268353
2,"""Sam Cooke""","""Ain't Misbehavin""",244730
3,"""Train""","""Hey_ Soul Sister""",209212
4,"""Lil Wayne / Eminem""","""Drop The World""",155717
…,…,…,…
45,"""The Black Keys""","""I'll Be Your Man""",28413
46,"""Niccolò Fabi""","""Costruire""",27766
47,"""Deepest Blue""","""Deepest Blue""",27004



  Collection: WAR


index,artist,title,play_count
u32,str,str,i64
0,"""Dwight Yoakam""","""You're The One""",726885
1,"""Björk""","""Undo""",648239
2,"""Sam Cooke""","""Ain't Misbehavin""",244730
3,"""Train""","""Hey_ Soul Sister""",209212
4,"""Pavement""","""Mercy:The Laundromat""",130116
…,…,…,…
45,"""Amy Winehouse""","""Me & Mr Jones""",25347
46,"""Alicia Keys""","""Empire State Of Mind (Part II)…",25203
47,"""Nirvana""","""Lithium""",25142



  Collection: HAPPINESS


...

index,artist,title,play_count
u32,str,str,i64
0,"""Dwight Yoakam""","""You're The One""",726885
1,"""Train""","""Hey_ Soul Sister""",209212
2,"""Cartola""","""Tive Sim""",185653
3,"""Lil Wayne / Eminem""","""Drop The World""",155717
4,"""Bill Withers""","""Make Love To Your Mind""",146978
…,…,…,…
45,"""Soundgarden""","""Burden In My Hand""",26013
46,"""Vanessa Williams""","""Colors Of The Wind""",26001
47,"""Nirvana""","""Lithium""",25142



  Collection: LONELINESS


index,artist,title,play_count
u32,str,str,i64
0,"""OneRepublic""","""Secrets""",292642
1,"""Five Iron Frenzy""","""Canada""",274627
2,"""Angels and Airwaves""","""The Gift""",192884
3,"""Cartola""","""Tive Sim""",185653
4,"""Randy Crawford""","""Almaz""",129868
…,…,…,…
45,"""Colbie Caillat""","""I Never Told You""",29113
46,"""Linkin Park""","""One Step Closer (Album Version…",28668
47,"""Avril Lavigne""","""My Happy Ending""",27203



  Collection: MONEY


index,artist,title,play_count
u32,str,str,i64
0,"""Dwight Yoakam""","""You're The One""",726885
1,"""Train""","""Marry Me""",174080
2,"""Pavement""","""Mercy:The Laundromat""",130116
3,"""Coldplay""","""The Scientist""",128837
4,"""Florence + The Machine""","""Cosmic Love""",94002
…,…,…,…
45,"""Linkin Park""","""One Step Closer (Album Version…",28668
46,"""The Black Keys""","""I'll Be Your Man""",28413
47,"""Metric""","""Gold Guns Girls""",28148


### Comparison

Check overlap between the three approaches for each keyword.

In [7]:
import polars as pl

def summarize(df):
    if df is None or len(df) == 0:
        return 0, set()
    return len(df), set(zip(df["artist"], df["title"]))

methods = [("baseline", baseline_results), ("w2v", w2v_results)]
methods += [(f"clf_{name}", clf_results[name]) for name in classifiers]

rows = {}
for kw in KEYWORDS:
    rows[kw.upper()] = {label: summarize(res.get(kw))[0] for label, res in methods}

print("Result sizes (rows per keyword and method):")
display(pl.DataFrame([{"keyword": kw, **vals} for kw, vals in rows.items()]))

Result sizes (rows per keyword and method):


keyword,baseline,w2v,clf_nb,clf_logistic,clf_sgd,clf_forest
str,i64,i64,i64,i64,i64,i64
"""LOVE""",50,50,50,50,50,50
"""WAR""",50,5,50,50,50,50
"""HAPPINESS""",50,50,50,50,50,50
"""LONELINESS""",50,3,50,50,50,50
"""MONEY""",50,50,50,50,50,50
